In [ ]:
import json
import pandas as pd
from pathlib import Path

from src.data_export import export_points, export_bmarkers

In [ ]:
root = Path("./raw_data")
dfs = []
for file in root.rglob("*.xlsx"):
    df = pd.read_excel(file, engine="openpyxl")
    df["source_file"] = str(file.relative_to(root))
    dfs.append(df)

combined_df = pd.concat(dfs, ignore_index=True)
combined_df

In [ ]:
# all areas in the WH
areas = combined_df.query("Name == 'WAYPOINT'")["ID"].to_list()
areas

In [ ]:
corners_df = (combined_df.groupby('source_file', as_index=False)[['Position X', 'Position Y']].agg(['min', 'max']))
corners_df

In [ ]:
corners_dict = {}

for  inx, row in corners_df.iterrows():
	corners_dict[areas[inx]] = {
        "corners": [
            {"x": int(row[('Position X', 'min')]), "y": int(row[('Position Y', 'min')]), "label": "Origin"},
            {"x": int(row[('Position X', 'max')])+1, "y": int(row[('Position Y', 'max')])+1, "label": "Max"}
            ]}

corners_dict

In [ ]:
bmarkers_df = combined_df[combined_df["Name"] == 'BMARKER']
bmarkers_df['unique_rack'] = bmarkers_df["ID"].apply(lambda x: x.split(".")[0])

In [ ]:
for area in areas:
    area_bms = bmarkers_df[bmarkers_df["source_file"].str.endswith(f"{area}.xlsx")]
    bmarkers_dict = export_bmarkers(area_bms,area)

    area_points = export_points(combined_df,area)

    combined = {**corners_dict[area], **bmarkers_dict, **area_points}
    with open(f"json_files\\mapping_{area}.json", "w", encoding="utf-8") as f:
        json.dump(combined, f, ensure_ascii=False, indent=2)


In [ ]:
corners_glob_df = combined_df[["Position X", "Position Y"]].agg(['min', 'max'])
corners_glob_df